# 00 — Ver os dados

Look at the raw data and at exactly what the model receives, before trusting any score.

Change `ESTACAO` and re-run. The zoom and leakage cells have their own dates to change.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT = Path.cwd().resolve()
for path in (PROJECT, PROJECT / "predict inflow"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pipeline_config import INTERIM_DIR, STATIONS
from previsao import (
    INICIO, NOWCAST_FEATURES, NOWCAST_ALTURA_FEATURES, PREVISAO_FEATURES,
    load_precip, split_train_test, training_table,
)

ESTACAO = "antanhol"  # ou "ameal"
DADOS_NOVOS = Path.home() / "Downloads" / "Dados de Projeto" / "dados-novos-pra-testar-no-modelo"
DIAMETRO = STATIONS[ESTACAO]["collector_diameter_m"]

## 1. All the data

Every 2-minute reading, from May 2025 to mid-October 2026, shown as hourly means so the plot stays light. The October readings come from the new folder (the model never trained on them). Gray band: before 1 July 2026, when the devices were not configured properly. The code drops all of it.

The bottom row is the share of readings in each hour where caudal is exactly 0.

In [2]:
def read_nivus(path):
    with path.open(encoding="utf-8-sig") as handle:
        header = next(i for i, line in enumerate(handle) if line.lower().startswith("date;time"))
    raw = pd.read_csv(path, sep=";", skiprows=header, encoding="utf-8-sig", dtype=str).dropna(subset=["Date", "Time"])
    out = pd.DataFrame({"datetime": pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")})
    for name, fragment in {"caudal_l_s": "app1_q", "altura_m": "app1_h", "velocidade_m_s": "app1_v"}.items():
        column = next(c for c in raw.columns if fragment in c.lower())
        out[name] = pd.to_numeric(raw[column].str.replace(",", ".", regex=False), errors="coerce")
    out["caudal_m3_h"] = out["caudal_l_s"] * 3.6
    return out


prefix = STATIONS[ESTACAO]["folder"].lower()
novos = pd.concat(
    [read_nivus(f) for folder in DADOS_NOVOS.iterdir() if folder.is_dir() and folder.name.lower().startswith(prefix)
     for f in sorted(folder.rglob("*")) if f.is_file() and f.parent.name.upper() == "DATA" and f.suffix.lower() == ".csv"],
    ignore_index=True,
)
antigos = pd.read_csv(INTERIM_DIR / f"{ESTACAO}_merged.csv", parse_dates=["datetime"],
                      usecols=["datetime", "caudal_l_s", "caudal_m3_h", "altura_m", "velocidade_m_s"])
leituras = (pd.concat([antigos, novos]).dropna(subset=["datetime"])
            .drop_duplicates("datetime", keep="last").sort_values("datetime").set_index("datetime"))
chuva = load_precip(ESTACAO)

por_hora = leituras.resample("h").agg({"altura_m": "mean", "velocidade_m_s": "mean", "caudal_m3_h": "mean"})
por_hora["zeros"] = leituras["caudal_l_s"].eq(0).resample("h").mean() * 100
print(f"{STATIONS[ESTACAO]['folder']}: {len(leituras):,} leituras de 2 min, "
      f"{leituras.index.min():%d/%m/%Y} → {leituras.index.max():%d/%m/%Y}")

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.025,
                    row_heights=[0.13, 0.22, 0.2, 0.3, 0.15],
                    subplot_titles=("Chuva (mm/h, Open-Meteo)", "Altura (m)", "Velocidade (m/s)",
                                    "Caudal (m³/h)", "% leituras com caudal = 0"))
fig.add_trace(go.Bar(x=chuva.index, y=chuva, marker_color="#378ADD", name="chuva"), row=1, col=1)
for row, col, color in ((2, "altura_m", "#0F6E56"), (3, "velocidade_m_s", "#BA7517"), (4, "caudal_m3_h", "#26215C"), (5, "zeros", "#A32D2D")):
    fig.add_trace(go.Scattergl(x=por_hora.index, y=por_hora[col], mode="lines", line=dict(width=1, color=color), name=col), row=row, col=1)
fig.add_hline(y=DIAMETRO, line=dict(color="gray", dash="dash"), row=2, col=1, annotation_text="diâmetro")
fig.add_vrect(x0=leituras.index.min(), x1=INICIO, fillcolor="gray", opacity=0.15, line_width=0)
fig.add_annotation(x=leituras.index.min(), y=1.04, xref="x", yref="paper", xanchor="left", showarrow=False,
                   text="cinzento: antes de 1/7/2026, configuração não fiável")
fig.update_layout(height=1000, template="plotly_white", showlegend=False, hovermode="x unified")
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.04), row=5, col=1)
fig.show()

Antanhol: 202,719 leituras de 2 min, 09/05/2025 → 14/10/2026


## 2. Zoom into the 2-minute readings

Pick any window. The default is the August 2026 storm.

In [3]:
ZOOM_INICIO, ZOOM_FIM = "2026-08-22", "2026-08-29"

w = leituras.loc[ZOOM_INICIO:ZOOM_FIM]
c = chuva.loc[ZOOM_INICIO:ZOOM_FIM]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.15, 0.25, 0.25, 0.35],
                    subplot_titles=("Chuva (mm/h)", "Altura (m)", "Velocidade (m/s)", "Caudal (m³/h)"))
fig.add_trace(go.Bar(x=c.index, y=c, marker_color="#378ADD"), row=1, col=1)
for row, col, color in ((2, "altura_m", "#0F6E56"), (3, "velocidade_m_s", "#BA7517"), (4, "caudal_m3_h", "#26215C")):
    fig.add_trace(go.Scattergl(x=w.index, y=w[col], mode="lines", line=dict(width=1, color=color)), row=row, col=1)
fig.update_layout(height=800, template="plotly_white", showlegend=False, hovermode="x unified")
fig.show()

## 3. What the model sees

`training_table` is the exact table the model is trained on: one row per hour, after the code's filters (from 1 July 2026, caudal above 0, height inside the pipe, at least 15 good readings in the hour). Each model only uses some of its columns, listed below. Blue: training hours. Orange: test hours. Hours missing from the table were filtered out.

In [4]:
tabela = training_table(ESTACAO)
treino, teste = split_train_test(tabela)
print(f"{len(tabela)} horas na tabela: {len(treino)} treino ({treino.index.min():%d/%m} → {treino.index.max():%d/%m}), "
      f"{len(teste)} teste ({teste.index.min():%d/%m} → {teste.index.max():%d/%m})")
print("alvo: caudal_total_m3_h (média do caudal na hora)")
print("modelo com altura:", NOWCAST_FEATURES)
print("modelo só altura: ", NOWCAST_ALTURA_FEATURES)
print("modelo só chuva:  ", PREVISAO_FEATURES)

completa = tabela.reindex(pd.date_range(tabela.index.min(), tabela.index.max(), freq="h"))
parte = pd.Series("treino", index=tabela.index).where(tabela.index < teste.index.min(), "teste")
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.4, 0.3, 0.3],
                    subplot_titles=("Alvo: caudal médio da hora (m³/h)", "Altura média e máxima da hora (m)",
                                    "Chuva das 3 h e das 6 h anteriores (mm)"))
for nome, cor in (("treino", "#185FA5"), ("teste", "#D85A30")):
    sel = completa["caudal_total_m3_h"].where(completa.index.isin(parte[parte == nome].index))
    fig.add_trace(go.Scatter(x=sel.index, y=sel, mode="lines", name=nome, line=dict(color=cor, width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=completa.index, y=completa["altura_media_m"], mode="lines", name="altura média", line=dict(color="#0F6E56", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=completa.index, y=completa["altura_max_m"], mode="lines", name="altura máxima", line=dict(color="#5DCAA5", width=1, dash="dot")), row=2, col=1)
fig.add_trace(go.Scatter(x=completa.index, y=completa["chuva_3h"], mode="lines", name="chuva 3 h", line=dict(color="#378ADD", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=completa.index, y=completa["chuva_6h"], mode="lines", name="chuva 6 h", line=dict(color="#7F77DD", width=1, dash="dot")), row=3, col=1)
fig.update_layout(height=800, template="plotly_white", hovermode="x unified")
fig.show()
tabela[["caudal_total_m3_h", *NOWCAST_FEATURES]].head(10)

1006 horas na tabela: 703 treino (22/07 → 20/08), 303 teste (21/08 → 02/09)
alvo: caudal_total_m3_h (média do caudal na hora)
modelo com altura: ['altura_media_m', 'altura_max_m', 'chuva_3h', 'chuva_6h', 'hour']
modelo só altura:  ['altura_media_m', 'altura_max_m']
modelo só chuva:   ['chuva_3h', 'chuva_6h', 'hour']


,caudal_total_m3_h,altura_media_m,altura_max_m,chuva_3h,chuva_6h,hour
datetime,,,,,,
2026-07-22 10:00:00,37.8444,0.039900,0.043,0.0,0.0,10
2026-07-22 11:00:00,38.7744,0.040333,0.044,0.0,0.0,11
2026-07-22 12:00:00,41.5848,0.041633,0.048,0.0,0.0,12
2026-07-22 13:00:00,37.3764,0.039667,0.042,0.0,0.0,13
2026-07-22 14:00:00,34.0596,0.037900,0.040,0.0,0.0,14
2026-07-22 15:00:00,36.9228,0.039400,0.042,0.0,0.0,15
2026-07-22 16:00:00,34.8732,0.038400,0.040,0.0,0.0,16
2026-07-22 17:00:00,30.7908,0.036000,0.038,0.0,0.0,17
2026-07-22 18:00:00,32.0028,0.036867,0.040,0.0,0.0,18


## 4. Leakage check for one hour

For the hour `HORA`, this rebuilds each value in the table from the raw readings and draws the time window each one comes from.

- Open-Meteo labels each hour's rain by its end: rain at 14:00 is what fell between 13:00 and 14:00.
- The table labels each hour by its start: caudal and height at 14:00 are the readings from 14:00 to 14:59.

The check passes if every input window ends no later than the end of the hour being predicted.

In [5]:
HORA = pd.Timestamp("2026-08-26 15:00")

h0, h1 = HORA, HORA + pd.Timedelta(hours=1)
linha = tabela.loc[HORA]
leit = leituras.loc[h0:h1 - pd.Timedelta(seconds=1)]
leit = leit[(leit["caudal_l_s"] > 0) & (leit["altura_m"] > 0) & (leit["altura_m"] < DIAMETRO)]
janelas = {
    "alvo: caudal médio": (h0, h1, leit["caudal_m3_h"].mean(), linha["caudal_total_m3_h"]),
    "altura média": (h0, h1, leit["altura_m"].mean(), linha["altura_media_m"]),
    "altura máxima": (h0, h1, leit["altura_m"].max(), linha["altura_max_m"]),
    # chuva_3h = chuva das etiquetas HORA-1h, HORA-2h, HORA-3h → caiu entre HORA-4h e HORA-1h
    "chuva 3 h": (h0 - pd.Timedelta(hours=4), h0 - pd.Timedelta(hours=1),
                  chuva.loc[h0 - pd.Timedelta(hours=3):h0 - pd.Timedelta(hours=1)].sum(), linha["chuva_3h"]),
    "chuva 6 h": (h0 - pd.Timedelta(hours=7), h0 - pd.Timedelta(hours=1),
                  chuva.loc[h0 - pd.Timedelta(hours=6):h0 - pd.Timedelta(hours=1)].sum(), linha["chuva_6h"]),
}
print(f"Hora {HORA:%d/%m/%Y %H:%M} ({'teste' if HORA >= teste.index.min() else 'treino'})")
for nome, (a, b, refeito, na_tabela) in janelas.items():
    ok = "ok" if np.isclose(refeito, na_tabela) else "DIFERENTE"
    print(f"  {nome:20s} {a:%d/%m %H:%M} → {b:%d/%m %H:%M}   refeito {refeito:8.3f}   na tabela {na_tabela:8.3f}   {ok}")
fim_alvo = h1
print("Nenhuma janela de entrada passa do fim da hora prevista:", all(b <= fim_alvo for a, b, *_ in janelas.values()))

fig = go.Figure()
for i, (nome, (a, b, *_)) in enumerate(janelas.items()):
    fig.add_trace(go.Scatter(x=[a, b], y=[nome, nome], mode="lines", line=dict(width=14, color="#D85A30" if i == 0 else "#185FA5"), showlegend=False))
fig.add_vline(x=h1, line=dict(color="black", dash="dash"), annotation_text="fim da hora prevista")
fig.update_layout(height=320, template="plotly_white", title="De onde vem cada valor (laranja = alvo, azul = entradas)",
                  xaxis=dict(range=[h0 - pd.Timedelta(hours=8), h1 + pd.Timedelta(hours=2)]))
fig.show()

Hora 26/08/2026 15:00 (teste)
  alvo: caudal médio   26/08 15:00 → 26/08 16:00   refeito   22.351   na tabela   22.351   ok
  altura média         26/08 15:00 → 26/08 16:00   refeito    0.031   na tabela    0.031   ok
  altura máxima        26/08 15:00 → 26/08 16:00   refeito    0.033   na tabela    0.033   ok
  chuva 3 h            26/08 11:00 → 26/08 14:00   refeito    2.200   na tabela    2.200   ok
  chuva 6 h            26/08 08:00 → 26/08 14:00   refeito   10.200   na tabela   10.200   ok
Nenhuma janela de entrada passa do fim da hora prevista: True


## 5. Height against caudal, by period

Each point is one hour. If one height always gave the same caudal, all the points would sit on one curve. Different colors on different curves mean the relationship changed between periods.

In [6]:
h = leituras[(leituras["caudal_l_s"] > 0) & (leituras["altura_m"] > 0) & (leituras["altura_m"] < DIAMETRO)]
h = h.resample("h").agg({"altura_m": "mean", "caudal_m3_h": "mean"}).dropna()
inicio_teste, inicio_outubro = teste.index.min(), pd.Timestamp("2026-10-01")
periodo = np.select(
    [h.index < INICIO, h.index < inicio_teste, h.index < inicio_outubro],
    ["antes de 1/7/2026 (não fiável)", "treino", "teste"], "outubro (nunca visto)")
fig = go.Figure()
for nome, cor in (("antes de 1/7/2026 (não fiável)", "#B4B2A9"), ("treino", "#185FA5"), ("teste", "#D85A30"), ("outubro (nunca visto)", "#0F6E56")):
    sel = h[periodo == nome]
    fig.add_trace(go.Scattergl(x=sel["altura_m"], y=sel["caudal_m3_h"], mode="markers", name=f"{nome} ({len(sel)} h)",
                               marker=dict(size=4, color=cor, opacity=0.6)))
fig.update_layout(height=600, template="plotly_white", xaxis_title="altura média da hora (m)", yaxis_title="caudal médio da hora (m³/h)")
fig.show()